In [1]:
import numpy as np

In [2]:
def generate_basis_vectors(theta):
    # Generate + and - basis vectors with respect to a given rotation in qubit space
    plus_theta = np.array([np.cos(theta/2), np.sin(theta/2)])
    minus_theta = np.array([np.sin(theta/2), -np.cos(theta/2)])

    return plus_theta, minus_theta

In [3]:
psi = (1/np.sqrt(2)) * np.array([0, 1, -1, 0])

In [4]:
# Angles over which to measure
a1 = 0
a2 = np.pi / 4
b1 = np.pi / 8
b2 = -np.pi / 8

In [60]:
# Generate the bases to measure with respect to
a_bases = [
    generate_basis_vectors(a1),
    generate_basis_vectors(a2)
]
a_basis = [0] * 2
a_basis[0], a_basis[1] = generate_basis_vectors(a1)

b_bases = [
    generate_basis_vectors(b1),
    generate_basis_vectors(b2)
]
b_basis = [0] * 2
b_basis[0], b_basis[1] = generate_basis_vectors(b1)

In [61]:
a_basis

[array([1., 0.]), array([ 0., -1.])]

In [62]:
b_basis

[array([0.98078528, 0.19509032]), array([ 0.19509032, -0.98078528])]

In [63]:
# Build the combined two-qubit bases for each combination of a & b bases
full_bases = [None] * 4
for i in range(2):
    for j in range(2):
        current_basis = [0] * 4
        dex = 0 

        # Combine every pair via kronecker product
        for a_v in a_bases[i]:
            for b_v in b_bases[j]:
                current_basis[dex] = np.kron(a_v, b_v)
                dex += 1
        
        full_bases[i*2 + j] = current_basis


full_basis = [0] * 4

dex = 0
for a_v in a_basis:
    for b_v in b_basis:
        full_basis[dex] = np.kron(a_v, b_v)
        dex += 1

full_basis

[array([0.98078528, 0.19509032, 0.        , 0.        ]),
 array([ 0.19509032, -0.98078528,  0.        , -0.        ]),
 array([ 0.        ,  0.        , -0.98078528, -0.19509032]),
 array([ 0.        , -0.        , -0.19509032,  0.98078528])]

In [64]:
full_bases

[[array([0.98078528, 0.19509032, 0.        , 0.        ]),
  array([ 0.19509032, -0.98078528,  0.        , -0.        ]),
  array([ 0.        ,  0.        , -0.98078528, -0.19509032]),
  array([ 0.        , -0.        , -0.19509032,  0.98078528])],
 [array([ 0.98078528, -0.19509032,  0.        , -0.        ]),
  array([-0.19509032, -0.98078528, -0.        , -0.        ]),
  array([ 0.        , -0.        , -0.98078528,  0.19509032]),
  array([-0.        , -0.        ,  0.19509032,  0.98078528])],
 [array([0.90612745, 0.18023996, 0.37533028, 0.07465783]),
  array([ 0.18023996, -0.90612745,  0.07465783, -0.37533028]),
  array([ 0.37533028,  0.07465783, -0.90612745, -0.18023996]),
  array([ 0.07465783, -0.37533028, -0.18023996,  0.90612745])],
 [array([ 0.90612745, -0.18023996,  0.37533028, -0.07465783]),
  array([-0.18023996, -0.90612745, -0.07465783, -0.37533028]),
  array([ 0.37533028, -0.07465783, -0.90612745,  0.18023996]),
  array([-0.07465783, -0.37533028,  0.18023996,  0.90612745]

In [65]:
probs = np.zeros(4)

for i in range(4):
    probs[i] = np.abs(np.dot(full_basis[i], psi))**2

probs

array([0.01903012, 0.48096988, 0.48096988, 0.01903012])

In [66]:
shots = 10000

sum = 0
for i in range(shots):
    choice = np.random.choice(4, p=probs)

    observed = (np.array([-1,-1]) ** np.unravel_index(choice, (2,2))).prod() # Product of observed values (either +- 1 each)

    sum += observed
sum /= shots
sum

# Result should be ~ -cos(a1 - b1)


np.float64(-0.9306)

In [67]:
# Empirically calculate S value. Note that classical theory predicts |S| <= 2
expected_values = [0] * 4
shot_distribution = [0] * 4
shots = 10000

prob_distributions = [None] * 4
for i in range(4):
    prob_distributions[i] = np.zeros(4)

    for j in range(4):
        prob_distributions[i][j] = np.abs(np.vdot(full_bases[i][j], psi))**2
    
for i in range(shots):
    basis_choice = np.random.choice(4)

    measured_state = np.random.choice(4, p=prob_distributions[basis_choice])

    observed = (np.array([-1,-1]) ** np.unravel_index(measured_state, (2,2))).prod()

    expected_values[basis_choice] += observed
    shot_distribution[basis_choice] += 1

for i in range(4):
    expected_values[i] = expected_values[i] / shot_distribution[i]

S = expected_values[0] - expected_values[1] + expected_values[2] + expected_values[3]
S

np.float64(-1.3114830280826502)

In [68]:
shot_distribution

[2559, 2457, 2411, 2573]

In [69]:
expected_values

[np.float64(-0.9171551387260649),
 np.float64(-0.9129019129019129),
 np.float64(-0.9228535877229366),
 np.float64(-0.3843762145355616)]

In [70]:
for theta in [a1, a2, b1, b2]:
    plus, minus = generate_basis_vectors(theta)
    print(np.dot(plus, minus))

0.0
2.1634997285925725e-17
2.72866517233415e-19
-2.72866517233415e-19


In [71]:
for i in range(4):
    print(i, np.sum(prob_distributions[i]))

0 0.9999999999999998
1 0.9999999999999998
2 0.9999999999999999
3 1.0


In [9]:
def calc_E(angl1, angl2, psi, shots=1000):
    base_1 = generate_basis_vectors(angl1)
    base_2 = generate_basis_vectors(angl2)

    prob_dist = np.zeros(4)
    basis_vects = []

    for v1 in base_1:
        for v2 in base_2:
            basis_vects.append(np.kron(v1, v2))
    
    for i in range(4):
        prob_dist[i] = np.abs(np.dot(basis_vects[i], psi))**2
    
    sum = 0
    for i in range(shots):
        choice = np.random.choice(4, p=prob_dist)

        observed = (np.array([-1,-1]) ** np.unravel_index(choice, (2,2))).prod()

        sum += observed
    sum /= shots

    return sum, -np.cos(angl1 - angl2)

In [45]:
calc_E(a1, b1, psi)[1] + calc_E(a1, b2, psi)[1] + calc_E(a2, b1, psi)[1] - calc_E(a2, b2, psi)[1]

np.float64(-2.8284271247461903)

In [42]:
a1, a2, b1, b2

(0, 1.5707963267948966, 0.7853981633974483, -0.7853981633974483)

In [43]:
a1 = 0
a2 = np.pi / 2
b1 = np.pi / 4
b2 = -np.pi / 4